In [1]:
import torch
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [3]:
from src.envs.agents.reinforce_agent import REINFORCEAgent
from src.game.template import calculate_output_np

In [4]:
# checkpoint_dir = '../output/2025-05-20/21:20:19.338686/agent1'

In [5]:
# checkpoint_dir = '../output/2025-05-22/02:06:48.278418/agent0'

In [6]:
checkpoint_dir = '../output/2025-05-22/02:06:48.278418/agent0'

In [7]:
info = {
    'train': True,
    'reinforce': True,
    'state_type': 'closest_ext',
    'gamma': 0.5,
    'action_space_n': 8,
}

In [8]:
del info['reinforce']

In [9]:
agent = REINFORCEAgent(**info)

In [10]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [11]:
data = {'entity_kind': [[1, 1, 3, 2, 0, 0, 0, 0, 0, 0]],
 'entity_features': [[[218.0, 3.0, 3.0, -3.0, 6.0, 0.0, 218.0],
   [221.0, 3.0, 1.0, -9.0, 10.0, 0.0, 221.0],
   [27.0, 0.0, 6.0, 6.0, 12.0, 0.0, 27.0],
   [2.0, -1.0, 7.0, 4.0, 11.0, 0.0, 2.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]]],
 'entity_dir': [[[0.0, 6, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 12, 0.0],
   [0.0, 12, 0.0, 0.0, 0.0],
   [0.0, 15, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0],
   [0.0, 0.0, 0.0, 0.0, 0.0]]]}

In [12]:
weights = {}
for k,v in agent.model.named_parameters():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

kind_embs.weight torch.Size([13, 32])
features_linear.weight torch.Size([32, 7])
features_linear.bias torch.Size([32])
dir_linear.weight torch.Size([16, 5])
dir_linear.bias torch.Size([16])
entity_linear.weight torch.Size([16, 80])
entity_linear.bias torch.Size([16])
entity_impact.weight torch.Size([8, 80])
entity_impact.bias torch.Size([8])
out_linear.weight torch.Size([1, 16])
out_linear.bias torch.Size([1])


In [13]:
calculate_output_np(data, weights, num_classes=8, softmax=True)

array([[7.59627917e-82, 3.50056444e-82, 7.43932712e-82, 7.41134817e-82,
        1.00094171e-81, 1.81639157e-81, 3.52144349e-82, 1.00000000e+00]])

In [14]:
tensor_data = {k: torch.tensor(v) for k,v in data.items()}

In [15]:
tensor_data = {
    'entity_kind': torch.IntTensor(data['entity_kind']),
    'entity_features': torch.FloatTensor(data['entity_features']),
    'entity_dir': torch.FloatTensor(data['entity_dir']),
}

In [16]:
model_output = agent.model(tensor_data)[0].detach().cpu().numpy()

In [17]:
model_output

array([0., 0., 0., 0., 0., 0., 0., 1.], dtype=float32)

In [18]:
weights['entity_impact.weight'].shape[0]

8

In [19]:
data2, data1 = zip(*weights.items())

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

In [20]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [21]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", "mode = 'dqn_ext'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        f.write(line)

In [22]:
!ls -lh ../src/game/template_submit.py

-rw-r--r--  1 aleksei  staff    28K 22 май 16:58 ../src/game/template_submit.py
